In [1]:
# import libraries
import polars as pl
from datetime import date
from tqdm.notebook import tqdm
import plotly.express as px

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(-1)
pl.Config.set_fmt_str_lengths(100)
pl.Config.set_tbl_width_chars(200)

# data path
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from src.config import RAW_DIR, INTERIM_DIR, PROCESSED_DIR, DOCS_DIR

# 2. Data Understanding

## 2.1. Objective and Summary

### - Objective

Define the dataset, understand its structure and quality, and establish the analytical population and feature eligibility for the credit risk modeling task.

### - Summary

**Dataset:** 2.93M loan records, 142 features, LendingClub 2007–2020 Q3.

**Data Quality**
- Pre-2016 null rates were volatile (28%–66%) and inconsistent (41%–70% of columns affected), reflecting platform data immaturity.
- From January 2016 onwards both metrics stabilised, establishing the analysis start date.
- Highest missingness is structural: hardship fields (93%–95%) and joint application fields (92%–93%), both excluded on leakage and relevance grounds.
- Remaining high-null columns within the January 2016 window will be dropped rather than imputed.

**Target Variable**
- loan_status consolidated from 11 raw values into three buckets: Paid (51.27%), Not Resolved (36.30%), Default (12.43%).
- The 12.43% headline default rate is diluted by the 36.30% unresolved share, concentrated in recent high-volume vintages.
- Restricting to resolved loans only shifts the effective default rate to 19.5%, the operationally relevant class imbalance figure.

**Modelling Sample**
- Filter 1: resolved loans only (Paid and Default), removing censored observations with unknown outcomes.
- Filter 2: issue_d from January 2016 onwards, aligning with the stable data recording window and consistent bureau attribute coverage.

**Feature Categorisation**
- Retain: origination features available at loan application time (credit score, income, DTI, grade, bureau attributes).
- Exclude: post-origination features (payment history, balances, recoveries) due to data leakage.
- Exclude: target and target-adjacent fields (loan_status and derived outcome variables).

## 2.2. Dataset Overview

* Dataset source and scope
* Number of observations and features
* Observation period
* Unit of observation
* Target candidate
* Key business context

In [2]:
# load data
lf = pl.scan_csv(
    RAW_DIR / "lending_club/Loan_status_2007-2020Q3.gzip",
    infer_schema_length=10000,
    ignore_errors=True)

In [3]:
# shape
lf.select([
    pl.len().alias("rows"),
    pl.lit(len(lf.collect_schema())).alias("columns") 
]).collect()

rows,columns
u32,i32
2925493,142


## 2.3. Data Structure

In [4]:
def create_data_profile(
    lf: pl.LazyFrame,
    description_df: pl.DataFrame,
    unique_sample_size: int = 5,
) -> pl.DataFrame:
    """
    Create a column-level data profile from a Polars LazyFrame.

    Includes:
    - feature
    - description
    - dtype
    - count
    - null_count
    - null_pct
    - unique_count
    - unique_sample
    - mean
    - median
    - mode

    The description is retrieved from a CSV containing:
    - LoanStatNew
    - Description

    If a feature does not have a matching description,
    the description is returned as null.

    Mean and median are only calculated for numeric columns.
    """

    # ---------------------------------------------------------
    # 1. Load feature descriptions
    # ---------------------------------------------------------

    description_df = (
        description_df
        .select([
            pl.col("LoanStatNew").alias("feature"),
            pl.col("Description").alias("description"),
        ])
        .with_columns(
            pl.col("feature").cast(pl.String),
            pl.col("description").cast(pl.String),
        )
        .unique(subset=["feature"])
    )

    # ---------------------------------------------------------
    # 2. Get schema
    # ---------------------------------------------------------

    schema = lf.collect_schema()

    profile = []

    # ---------------------------------------------------------
    # 3. Profile each feature
    # ---------------------------------------------------------

    for col, dtype in tqdm(schema.items()):

        stats = (
            lf.select([
                pl.len().alias("count"),
                pl.col(col).null_count().alias("null_count"),
                pl.col(col).n_unique().alias("unique_count"),
            ])
            .collect()
            .row(0)
        )

        count, null_count, unique_count = stats

        null_pct = (
            null_count / count * 100
            if count > 0
            else 0
        )

        # -----------------------------------------------------
        # Unique sample
        # -----------------------------------------------------

        unique_sample = (
            lf.select(
                pl.col(col)
                .drop_nulls()
                .unique()
                .head(unique_sample_size)
                .alias(col)
            )
            .collect()
            .get_column(col)
            .to_list()
        )

        # -----------------------------------------------------
        # Default statistics
        # -----------------------------------------------------

        mean = None
        median = None
        mode = None

        # -----------------------------------------------------
        # Numeric statistics
        # -----------------------------------------------------

        if dtype.is_numeric():

            numeric_stats = (
                lf.select([
                    pl.col(col).mean().alias("mean"),
                    pl.col(col).median().alias("median"),
                ])
                .collect()
                .row(0)
            )

            mean, median = numeric_stats

        # -----------------------------------------------------
        # Mode
        # -----------------------------------------------------

        mode_result = (
            lf.select(
                pl.col(col)
                .drop_nulls()
                .mode()
                .head(1)
                .alias("mode")
            )
            .collect()
            .get_column("mode")
            .to_list()
        )

        if mode_result:
            mode = mode_result[0]

        # -----------------------------------------------------
        # Store profile
        # -----------------------------------------------------

        profile.append({
            "feature": col,
            "dtype": str(dtype),
            "count": count,
            "null_count": null_count,
            "null_pct": null_pct,
            "unique_count": unique_count,
            "unique_sample": unique_sample,
            "mean": mean,
            "median": median,
            "mode": mode,
        })

    # ---------------------------------------------------------
    # 4. Create profile DataFrame
    # ---------------------------------------------------------

    profile_df = pl.DataFrame(profile)

    # ---------------------------------------------------------
    # 5. Add descriptions
    # ---------------------------------------------------------

    profile_df = (
        profile_df
        .join(
            description_df,
            on="feature",
            how="left",
        )
        .select([
            "feature",
            "description",
            "dtype",
            "count",
            "null_count",
            "null_pct",
            "unique_count",
            "unique_sample",
            "mean",
            "median",
            "mode",
        ])
    )

    return profile_df

In [5]:
# # create data profile
# description_df = pl.read_excel(RAW_DIR / "lending_club/LCDataDictionary.xlsx")
# data_profile = create_data_profile(lf, description_df)

# # save data profile to parquet
# data_profile.write_parquet(INTERIM_DIR / "data_profile_raw.parquet")

# load data profile
data_profile = pl.read_parquet(INTERIM_DIR / "data_profile_raw.parquet")

In [6]:
# prettify data display
from great_tables import GT
from IPython.display import HTML, display

table = (GT(data_profile).tab_header(title="Dataset Profile Summary"))

html = table._repr_html_()

display(HTML(f"""
<div style="
    max-height: 600px;
    overflow-y: auto;
    overflow-x: auto;
    border: 1px solid #ddd;
">
    {html}
</div>
"""))

In [7]:
# sample data
lf.head(5).collect()

,id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,url,purpose,title,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,last_fico_range_high,last_fico_range_low,collections_12_mths_ex_med,mths_since_last_major_derog,policy_code,application_type,annual_inc_joint,dti_joint,verification_status_joint,acc_now_delinq,tot_coll_amt,tot_cur_bal,open_acc_6m,open_act_il,open_il_12m,open_il_24m,mths_since_rcnt_il,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m,acc_open_past_24mths,avg_cur_bal,bc_open_to_buy,bc_util,chargeoff_within_12_mths,delinq_amnt,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,mths_since_recent_bc_dlq,mths_since_recent_inq,mths_since_recent_revol_delinq,num_accts_ever_120_pd,num_actv_bc_tl,num_actv_rev_tl,num_bc_sats,num_bc_tl,num_il_tl,num_op_rev_tl,num_rev_accts,num_rev_tl_bal_gt_0,num_sats,num_tl_120dpd_2m,num_tl_30dpd,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,revol_bal_joint,sec_app_fico_range_low,sec_app_fico_range_high,sec_app_earliest_cr_line,sec_app_inq_last_6mths,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,sec_app_chargeoff_within_12_mths,sec_app_collections_12_mths_ex_med,hardship_flag,hardship_type,hardship_reason,hardship_status,deferral_term,hardship_amount,hardship_start_date,hardship_end_date,payment_plan_start_date,hardship_length,hardship_dpd,hardship_loan_status,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount,debt_settlement_flag
i64,i64,i64,i64,f64,str,str,f64,str,str,str,str,str,f64,str,str,str,str,str,str,str,str,str,f64,i64,str,i64,i64,i64,i64,i64,i64,i64,i64,str,i64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,str,str,i64,i64,i64,str,i64,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
0,1077501,5000,5000,4975.0,""" 36 months""",""" 10.65%""",162.87,"""B""","""B2""",null,"""10+ years""","""RENT""",24000.0,"""Verified""","""Dec-2011""","""Fully Paid""","""n""","""https://lendingclub.com/browse/loanDetail.action?loan_id=1077501""","""credit_card""","""Computer""","""860xx""","""AZ""",27.65,0,"""Jan-1985""",735,739,1,null,null,3,0,13648,"""83.7%""",9,"""f""",0.0,0.0,5863.155187,5833.84,5000.0,863.16,0.0,0.0,0.0,"""Jan-2015""",171.62,null,"""May-2020""",704,700,0,null,1,"""Individual""",null,null,null,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""N""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""N"""
1,1077430,2500,2500,2500.0,""" 60 months""",""" 15.27%""",59.83,"""C""","""C4""","""Ryder""","""< 1 year""","""RENT""",30000.0,"""Source Verified""","""Dec-2011""","""Charged Off""","""n""","""https://lendingclub.com/browse/loanDetail.action?loan_id=1077430""","""car"

## 2.4. Feature Catalogue

The dataset contains a large number of variables covering borrower characteristics, credit behavior, loan attributes, and loan performance. To support downstream modeling and prevent inappropriate feature usage, each variable is classified using several complementary dimensions.

The catalogue separates **what a feature represents**, **when the information becomes available**, and **whether it can be used for the origination-time Probability of Default (PD) model**.

The following attributes are defined for each feature:

| Attribute         | Definition                                                                                                |
| ----------------- | --------------------------------------------------------------------------------------------------------- |
| `category`      | Business domain represented by the feature                                                                |
| `semantic_type` | Analytical meaning and expected data representation                                                       |
| `availability`  | Point in the lending lifecycle when the information is available                                          |
| `pd_eligible`   | Whether the feature is eligible for the origination-time PD model                                         |
| `leakage_risk`  | Risk that the feature contains information unavailable at prediction time or derived from future outcomes |
| `dtype`         | Physical data type currently stored in the dataset                                                        |

### 2.4.1. Feature Category

Features are grouped based on their primary **business meaning and role in describing credit risk**.

| Category             | Definition                                                            |
| -------------------- | --------------------------------------------------------------------- |
| `borrower_profile`   | Borrower characteristics such as income, employment, and housing.     |
| `credit_score`       | Credit score and FICO-related information.                            |
| `credit_history`     | Age, depth, and historical structure of credit accounts.              |
| `credit_profile`     | Overall current credit position and aggregate credit exposure.        |
| `revolving_credit`   | Revolving credit balances, limits, and utilization.                   |
| `installment_credit` | Installment credit accounts, balances, limits, and activity.          |
| `delinquency`        | Current or historical delinquency and past-due behavior.              |
| `payment_history`    | Loan payment and repayment performance.                               |
| `public_record`      | Bankruptcies, tax liens, and other public credit records.             |
| `credit_inquiry`     | Recent credit inquiries and credit-seeking activity.                  |
| `loan_info`          | Loan amount, term, purpose, pricing, and application characteristics. |
| `collection`         | Collection activity and recovery information.                         |
| `hardship`           | Hardship programs, payment assistance, and related information.       |
| `loan_status`        | Loan status and outcome information used for target construction.     |
| `metadata`           | Identifiers and technical fields used for record management.          |

**Assignment principle:** Each feature is assigned to one primary category based on its business meaning. Data type, availability, PD eligibility, and leakage risk are assessed separately.



In [8]:
# mapping category

CATEGORY_FEATURE_MAPPING ={
    '': 'metadata', 
    'id': 'metadata', 
    'loan_amnt': 'loan_info', 
    'funded_amnt': 'loan_info', 
    'funded_amnt_inv': 'loan_info', 
    'term': 'loan_info', 
    'int_rate': 'loan_info', 
    'installment': 'loan_info', 
    'grade': 'credit_profile', 
    'sub_grade': 'credit_profile', 
    'emp_title': 'borrower_profile', 
    'emp_length': 'borrower_profile', 
    'home_ownership': 'borrower_profile', 
    'annual_inc': 'borrower_profile', 
    'verification_status': 'borrower_profile', 
    'issue_d': 'loan_info', 
    'loan_status': 'loan_status', 
    'pymnt_plan': 'loan_status', 
    'url': 'metadata', 
    'purpose': 'loan_info', 
    'title': 'loan_info', 
    'zip_code': 'borrower_profile', 
    'addr_state': 'borrower_profile', 
    'dti': 'credit_profile', 
    'delinq_2yrs': 'delinquency', 
    'earliest_cr_line': 'credit_history', 
    'fico_range_low': 'credit_score', 
    'fico_range_high': 'credit_score', 
    'inq_last_6mths': 'credit_inquiry', 
    'mths_since_last_delinq': 'delinquency', 
    'mths_since_last_record': 'public_record', 
    'open_acc': 'credit_profile', 
    'pub_rec': 'public_record', 
    'revol_bal': 'revolving_credit', 
    'revol_util': 'revolving_credit', 
    'total_acc': 'credit_profile', 
    'initial_list_status': 'loan_info', 
    'out_prncp': 'payment_history', 
    'out_prncp_inv': 'payment_history', 
    'total_pymnt': 'payment_history', 
    'total_pymnt_inv': 'payment_history', 
    'total_rec_prncp': 'payment_history', 
    'total_rec_int': 'payment_history', 
    'total_rec_late_fee': 'payment_history', 
    'recoveries': 'payment_history', 
    'collection_recovery_fee': 'payment_history', 
    'last_pymnt_d': 'payment_history', 
    'last_pymnt_amnt': 'payment_history', 
    'next_pymnt_d': 'payment_history', 
    'last_credit_pull_d': 'credit_history', 
    'last_fico_range_high': 'credit_score', 
    'last_fico_range_low': 'credit_score', 
    'collections_12_mths_ex_med': 'delinquency', 
    'mths_since_last_major_derog': 'delinquency', 
    'policy_code': 'metadata', 
    'application_type': 'loan_info', 
    'annual_inc_joint': 'borrower_profile', 
    'dti_joint': 'credit_profile', 
    'verification_status_joint': 'borrower_profile', 
    'acc_now_delinq': 'delinquency', 
    'tot_coll_amt': 'collection', 
    'tot_cur_bal': 'credit_profile', 
    'open_acc_6m': 'credit_profile', 
    'open_act_il': 'installment_credit', 
    'open_il_12m': 'installment_credit', 
    'open_il_24m': 'installment_credit', 
    'mths_since_rcnt_il': 'installment_credit', 
    'total_bal_il': 'installment_credit', 
    'il_util': 'installment_credit', 
    'open_rv_12m': 'revolving_credit', 
    'open_rv_24m': 'revolving_credit', 
    'max_bal_bc': 'revolving_credit', 
    'all_util': 'credit_profile', 
    'total_rev_hi_lim': 'revolving_credit', 
    'inq_fi': 'credit_inquiry', 
    'total_cu_tl': 'credit_profile', 
    'inq_last_12m': 'credit_inquiry', 
    'acc_open_past_24mths': 'credit_history', 
    'avg_cur_bal': 'credit_profile', 
    'bc_open_to_buy': 'revolving_credit', 
    'bc_util': 'revolving_credit', 
    'chargeoff_within_12_mths': 'delinquency', 
    'delinq_amnt': 'delinquency', 
    'mo_sin_old_il_acct': 'installment_credit', 
    'mo_sin_old_rev_tl_op': 'revolving_credit', 
    'mo_sin_rcnt_rev_tl_op': 'revolving_credit', 
    'mo_sin_rcnt_tl': 'credit_history', 
    'mort_acc': 'credit_profile', 
    'mths_since_recent_bc': 'revolving_credit', 
    'mths_since_recent_bc_dlq': 'delinquency', 
    'mths_since_recent_inq': 'credit_inquiry', 
    'mths_since_recent_revol_delinq': 'delinquency', 
    'num_accts_ever_120_pd': 'delinquency', 
    'num_actv_bc_tl': 'revolving_credit', 
    'num_actv_rev_tl': 'revolving_credit', 
    'num_bc_sats': 'revolving_credit', 
    'num_bc_tl': 'revolving_credit', 
    'num_il_tl': 'installment_credit', 
    'num_op_rev_tl': 'revolving_credit', 
    'num_rev_accts': 'revolving_credit', 
    'num_rev_tl_bal_gt_0': 'revolving_credit', 
    'num_sats': 'credit_profile', 
    'num_tl_120dpd_2m': 'delinquency', 
    'num_tl_30dpd': 'delinquency', 
    'num_tl_90g_dpd_24m': 'delinquency', 
    'num_tl_op_past_12m': 'credit_history', 
    'pct_tl_nvr_dlq': 'credit_profile', 
    'percent_bc_gt_75': 'revolving_credit', 
    'pub_rec_bankruptcies': 'public_record', 
    'tax_liens': 'public_record', 
    'tot_hi_cred_lim': 'credit_profile', 
    'total_bal_ex_mort': 'credit_profile', 
    'total_bc_limit': 'revolving_credit', 
    'total_il_high_credit_limit': 'installment_credit', 
    'revol_bal_joint': 'revolving_credit', 
    'sec_app_fico_range_low': 'credit_score', 
    'sec_app_fico_range_high': 'credit_score', 
    'sec_app_earliest_cr_line': 'credit_history', 
    'sec_app_inq_last_6mths': 'credit_inquiry', 
    'sec_app_mort_acc': 'credit_profile', 
    'sec_app_open_acc': 'credit_profile', 
    'sec_app_revol_util': 'revolving_credit', 
    'sec_app_open_act_il': 'installment_credit', 
    'sec_app_num_rev_accts': 'revolving_credit', 
    'sec_app_chargeoff_within_12_mths': 'delinquency', 
    'sec_app_collections_12_mths_ex_med': 'collection', 
    'hardship_flag': 'hardship', 
    'hardship_type': 'hardship', 
    'hardship_reason': 'hardship', 
    'hardship_status': 'hardship', 
    'deferral_term': 'hardship', 
    'hardship_amount': 'hardship', 
    'hardship_start_date': 'hardship', 
    'hardship_end_date': 'hardship', 
    'payment_plan_start_date': 'hardship', 
    'hardship_length': 'hardship', 
    'hardship_dpd': 'hardship', 
    'hardship_loan_status': 'hardship', 
    'orig_projected_additional_accrued_interest': 'hardship', 
    'hardship_payoff_balance_amount': 'hardship', 
    'hardship_last_payment_amount': 'hardship', 
    'debt_settlement_flag': 'loan_status'
}

In [9]:
data_profile = (
    data_profile
    .with_columns(
        pl.col("feature").replace(CATEGORY_FEATURE_MAPPING).alias("category")
    )
    .select([
        "feature",
        "description",
        "category",
        "dtype",
        "count",
        "null_count",
        "null_pct",
        "unique_count",
        "unique_sample",
        "mean",
        "median",
        "mode",
    ])
)

data_profile.head(5)

feature,description,category,dtype,count,null_count,null_pct,unique_count,unique_sample,mean,median,mode
str,str,str,str,i64,i64,f64,i64,list[str],f64,f64,str
"""""",null,"""metadata""","""Int64""",2925493,0,0.0,421095,"[""0"", ""1"", … ""4""]",86974.19494,68925.0,"""8944"""
"""id""","""A unique LC assigned ID for the loan listing.""","""metadata""","""Int64""",2925493,1,0.000034,2925493,"[""54734"", ""55521"", … ""56121""]",9.7830e7,1.0761e8,"""141921161"""
"""loan_amnt""","""The listed amount of the loan applied for by the borrower. If at some point in time, the credit depa…","""loan_info""","""Int64""",2925493,1,0.000034,1573,"[""500"", ""550"", … ""725""]",15358.775105,13000.0,"""10000"""
"""funded_amnt""","""The total amount committed to that loan at that point in time.""","""loan_info""","""Int64""",2925493,1,0.000034,1573,"[""500"", ""550"", … ""725""]",15354.704909,13000.0,"""10000"""
"""funded_amnt_inv""","""The total amount committed by investors for that loan at that point in time.""","""loan_info""","""Float64""",2925493,1,0.000034,10064,"[""0.0"", ""0.000121098108"", … ""0.000531133069""]",15340.046287,13000.0,"""10000"""


### 2.4.2. Semantic Type

`semantic_type` describes the **analytical meaning and expected representation** of a feature, independent of its current physical dtype.

| Semantic Type | Definition                                                                        |
| ------------- | --------------------------------------------------------------------------------- |
| `numeric`     | Continuous numerical measurement used for quantitative analysis.                  |
| `count`       | Non-negative integer representing the number of events, accounts, or occurrences. |
| `monetary`    | Numerical value representing an amount of money.                                  |
| `percentage`  | Numerical value representing a rate or proportion.                                |
| `categorical` | Discrete values representing distinct groups or classes.                          |
| `ordinal`     | Categorical values with a meaningful order or ranking.                            |
| `date`        | Calendar date or time-related information.                                        |
| `boolean`     | Binary indicator representing two states.                                         |
| `identifier`  | Value used to uniquely identify or reference a record.                            |
| `text`        | Free-form textual information.                                                    |
| `target`      | Variable representing the outcome to be predicted.                                |

**Assignment principle:** Semantic type is determined by what the feature represents analytically, not by how it is currently stored. For example, `int_rate` stored as `String` is still semantically a `percentage`, while `issue_d` is semantically a `date`.


In [10]:
# mapping semantic type

SEMANTIC_TYPE_MAPPING ={
    '': 'count', 
    'id': 'identifier', 
    'loan_amnt': 'monetary', 
    'funded_amnt': 'monetary', 
    'funded_amnt_inv': 'monetary', 
    'term': 'ordinal', 
    'int_rate': 'percentage', 
    'installment': 'monetary', 
    'grade': 'ordinal', 
    'sub_grade': 'ordinal', 
    'emp_title': 'text', 
    'emp_length': 'ordinal', 
    'home_ownership': 'categorical', 
    'annual_inc': 'monetary', 
    'verification_status': 'categorical', 
    'issue_d': 'date', 
    'loan_status': 'target', 
    'pymnt_plan': 'boolean', 
    'url': 'identifier', 
    'purpose': 'categorical', 
    'title': 'text', 
    'zip_code': 'categorical', 
    'addr_state': 'categorical', 
    'dti': 'percentage', 
    'delinq_2yrs': 'count', 
    'earliest_cr_line': 'date', 
    'fico_range_low': 'numeric', 
    'fico_range_high': 'numeric', 
    'inq_last_6mths': 'count', 
    'mths_since_last_delinq': 'numeric', 
    'mths_since_last_record': 'numeric', 
    'open_acc': 'count', 
    'pub_rec': 'count', 
    'revol_bal': 'monetary', 
    'revol_util': 'percentage', 
    'total_acc': 'count', 
    'initial_list_status': 'categorical', 
    'out_prncp': 'monetary', 
    'out_prncp_inv': 'monetary', 
    'total_pymnt': 'monetary', 
    'total_pymnt_inv': 'monetary', 
    'total_rec_prncp': 'monetary', 
    'total_rec_int': 'monetary', 
    'total_rec_late_fee': 'monetary', 
    'recoveries': 'monetary', 
    'collection_recovery_fee': 'monetary', 
    'last_pymnt_d': 'date', 
    'last_pymnt_amnt': 'monetary', 
    'next_pymnt_d': 'date', 
    'last_credit_pull_d': 'date', 
    'last_fico_range_high': 'numeric', 
    'last_fico_range_low': 'numeric', 
    'collections_12_mths_ex_med': 'count', 
    'mths_since_last_major_derog': 'numeric', 
    'policy_code': 'categorical', 
    'application_type': 'categorical', 
    'annual_inc_joint': 'monetary', 
    'dti_joint': 'percentage', 
    'verification_status_joint': 'categorical', 
    'acc_now_delinq': 'count', 
    'tot_coll_amt': 'monetary', 
    'tot_cur_bal': 'monetary', 
    'open_acc_6m': 'count', 
    'open_act_il': 'count', 
    'open_il_12m': 'count', 
    'open_il_24m': 'count', 
    'mths_since_rcnt_il': 'numeric', 
    'total_bal_il': 'monetary', 
    'il_util': 'percentage', 
    'open_rv_12m': 'count', 
    'open_rv_24m': 'count', 
    'max_bal_bc': 'monetary', 
    'all_util': 'percentage', 
    'total_rev_hi_lim': 'monetary', 
    'inq_fi': 'count', 
    'total_cu_tl': 'count', 
    'inq_last_12m': 'count', 
    'acc_open_past_24mths': 'count', 
    'avg_cur_bal': 'monetary', 
    'bc_open_to_buy': 'monetary', 
    'bc_util': 'percentage', 
    'chargeoff_within_12_mths': 'count', 
    'delinq_amnt': 'monetary', 
    'mo_sin_old_il_acct': 'numeric', 
    'mo_sin_old_rev_tl_op': 'numeric', 
    'mo_sin_rcnt_rev_tl_op': 'numeric', 
    'mo_sin_rcnt_tl': 'numeric', 
    'mort_acc': 'count', 
    'mths_since_recent_bc': 'numeric', 
    'mths_since_recent_bc_dlq': 'numeric', 
    'mths_since_recent_inq': 'numeric', 
    'mths_since_recent_revol_delinq': 'numeric', 
    'num_accts_ever_120_pd': 'count', 
    'num_actv_bc_tl': 'count', 
    'num_actv_rev_tl': 'count', 
    'num_bc_sats': 'count', 
    'num_bc_tl': 'count', 
    'num_il_tl': 'count', 
    'num_op_rev_tl': 'count', 
    'num_rev_accts': 'count', 
    'num_rev_tl_bal_gt_0': 'count', 
    'num_sats': 'count', 
    'num_tl_120dpd_2m': 'count', 
    'num_tl_30dpd': 'count', 
    'num_tl_90g_dpd_24m': 'count', 
    'num_tl_op_past_12m': 'count', 
    'pct_tl_nvr_dlq': 'percentage', 
    'percent_bc_gt_75': 'percentage', 
    'pub_rec_bankruptcies': 'count', 
    'tax_liens': 'count', 
    'tot_hi_cred_lim': 'monetary', 
    'total_bal_ex_mort': 'monetary', 
    'total_bc_limit': 'monetary', 
    'total_il_high_credit_limit': 'monetary', 
    'revol_bal_joint': 'monetary', 
    'sec_app_fico_range_low': 'numeric', 
    'sec_app_fico_range_high': 'numeric', 
    'sec_app_earliest_cr_line': 'date', 
    'sec_app_inq_last_6mths': 'count', 
    'sec_app_mort_acc': 'count', 
    'sec_app_open_acc': 'count', 
    'sec_app_revol_util': 'percentage', 
    'sec_app_open_act_il': 'count', 
    'sec_app_num_rev_accts': 'count', 
    'sec_app_chargeoff_within_12_mths': 'count', 
    'sec_app_collections_12_mths_ex_med': 'count', 
    'hardship_flag': 'boolean', 
    'hardship_type': 'categorical', 
    'hardship_reason': 'categorical', 
    'hardship_status': 'categorical', 
    'deferral_term': 'count', 
    'hardship_amount': 'monetary', 
    'hardship_start_date': 'date', 
    'hardship_end_date': 'date', 
    'payment_plan_start_date': 'date', 
    'hardship_length': 'count', 
    'hardship_dpd': 'count', 
    'hardship_loan_status': 'categorical', 
    'orig_projected_additional_accrued_interest': 'monetary', 
    'hardship_payoff_balance_amount': 'monetary', 
    'hardship_last_payment_amount': 'monetary', 
    'debt_settlement_flag': 'boolean'
}

In [11]:
data_profile = (
    data_profile
    .with_columns(
        pl.col("feature").replace(SEMANTIC_TYPE_MAPPING).alias("semantic_type")
    )
    .select([
        "feature",
        "description",
        "category",
        "semantic_type",
        "dtype",
        "count",
        "null_count",
        "null_pct",
        "unique_count",
        "unique_sample",
        "mean",
        "median",
        "mode",
    ])
)

data_profile.head(5)

feature,description,category,semantic_type,dtype,count,null_count,null_pct,unique_count,unique_sample,mean,median,mode
str,str,str,str,str,i64,i64,f64,i64,list[str],f64,f64,str
"""""",null,"""metadata""","""count""","""Int64""",2925493,0,0.0,421095,"[""0"", ""1"", … ""4""]",86974.19494,68925.0,"""8944"""
"""id""","""A unique LC assigned ID for the loan listing.""","""metadata""","""identifier""","""Int64""",2925493,1,0.000034,2925493,"[""54734"", ""55521"", … ""56121""]",9.7830e7,1.0761e8,"""141921161"""
"""loan_amnt""","""The listed amount of the loan applied for by the borrower. If at some point in time, the credit depa…","""loan_info""","""monetary""","""Int64""",2925493,1,0.000034,1573,"[""500"", ""550"", … ""725""]",15358.775105,13000.0,"""10000"""
"""funded_amnt""","""The total amount committed to that loan at that point in time.""","""loan_info""","""monetary""","""Int64""",2925493,1,0.000034,1573,"[""500"", ""550"", … ""725""]",15354.704909,13000.0,"""10000"""
"""funded_amnt_inv""","""The total amount committed by investors for that loan at that point in time.""","""loan_info""","""monetary""","""Float64""",2925493,1,0.000034,10064,"[""0.0"", ""0.000121098108"", … ""0.000531133069""]",15340.046287,13000.0,"""10000"""


### 2.4.3. Feature Availability

`availability` identifies **when the information becomes available relative to loan origination and subsequent loan performance**.

| Availability       | Definition                                                                       |
| ------------------ | -------------------------------------------------------------------------------- |
| `origination`      | Information available at or before the loan application or origination decision. |
| `post_origination` | Information generated or updated after the loan is originated.                   |
| `outcome`          | Information representing the eventual loan outcome or realized performance.      |

**Assignment principle:** Classification is based on the timing of the information, not whether the feature appears predictive.


In [12]:
# mapping availability

AVAILABILITY_MAPPING ={
    '': 'origination', 
    'id': 'origination', 
    'loan_amnt': 'origination', 
    'funded_amnt': 'origination', 
    'funded_amnt_inv': 'origination', 
    'term': 'origination', 
    'int_rate': 'origination', 
    'installment': 'origination', 
    'grade': 'origination', 
    'sub_grade': 'origination', 
    'emp_title': 'origination', 
    'emp_length': 'origination', 
    'home_ownership': 'origination', 
    'annual_inc': 'origination', 
    'verification_status': 'origination', 
    'issue_d': 'origination', 
    'loan_status': 'outcome', 
    'pymnt_plan': 'post_origination', 
    'url': 'origination', 
    'purpose': 'origination', 
    'title': 'origination', 
    'zip_code': 'origination', 
    'addr_state': 'origination', 
    'dti': 'origination', 
    'delinq_2yrs': 'origination', 
    'earliest_cr_line': 'origination', 
    'fico_range_low': 'origination', 
    'fico_range_high': 'origination', 
    'inq_last_6mths': 'origination', 
    'mths_since_last_delinq': 'origination', 
    'mths_since_last_record': 'origination', 
    'open_acc': 'origination', 
    'pub_rec': 'origination', 
    'revol_bal': 'origination', 
    'revol_util': 'origination', 
    'total_acc': 'origination', 
    'initial_list_status': 'origination', 
    'out_prncp': 'post_origination', 
    'out_prncp_inv': 'post_origination', 
    'total_pymnt': 'post_origination', 
    'total_pymnt_inv': 'post_origination', 
    'total_rec_prncp': 'post_origination', 
    'total_rec_int': 'post_origination', 
    'total_rec_late_fee': 'post_origination', 
    'recoveries': 'post_origination', 
    'collection_recovery_fee': 'post_origination', 
    'last_pymnt_d': 'post_origination', 
    'last_pymnt_amnt': 'post_origination', 
    'next_pymnt_d': 'post_origination', 
    'last_credit_pull_d': 'post_origination', 
    'last_fico_range_high': 'post_origination', 
    'last_fico_range_low': 'post_origination', 
    'collections_12_mths_ex_med': 'origination', 
    'mths_since_last_major_derog': 'origination', 
    'policy_code': 'origination', 
    'application_type': 'origination', 
    'annual_inc_joint': 'origination', 
    'dti_joint': 'origination', 
    'verification_status_joint': 'origination', 
    'acc_now_delinq': 'origination', 
    'tot_coll_amt': 'origination', 
    'tot_cur_bal': 'origination', 
    'open_acc_6m': 'origination', 
    'open_act_il': 'origination', 
    'open_il_12m': 'origination', 
    'open_il_24m': 'origination', 
    'mths_since_rcnt_il': 'origination', 
    'total_bal_il': 'origination', 
    'il_util': 'origination', 
    'open_rv_12m': 'origination', 
    'open_rv_24m': 'origination', 
    'max_bal_bc': 'origination', 
    'all_util': 'origination', 
    'total_rev_hi_lim': 'origination', 
    'inq_fi': 'origination', 
    'total_cu_tl': 'origination', 
    'inq_last_12m': 'origination', 
    'acc_open_past_24mths': 'origination', 
    'avg_cur_bal': 'origination', 
    'bc_open_to_buy': 'origination', 
    'bc_util': 'origination', 
    'chargeoff_within_12_mths': 'origination', 
    'delinq_amnt': 'origination', 
    'mo_sin_old_il_acct': 'origination', 
    'mo_sin_old_rev_tl_op': 'origination', 
    'mo_sin_rcnt_rev_tl_op': 'origination', 
    'mo_sin_rcnt_tl': 'origination', 
    'mort_acc': 'origination', 
    'mths_since_recent_bc': 'origination', 
    'mths_since_recent_bc_dlq': 'origination', 
    'mths_since_recent_inq': 'origination', 
    'mths_since_recent_revol_delinq': 'origination', 
    'num_accts_ever_120_pd': 'origination', 
    'num_actv_bc_tl': 'origination', 
    'num_actv_rev_tl': 'origination', 
    'num_bc_sats': 'origination', 
    'num_bc_tl': 'origination', 
    'num_il_tl': 'origination', 
    'num_op_rev_tl': 'origination', 
    'num_rev_accts': 'origination', 
    'num_rev_tl_bal_gt_0': 'origination', 
    'num_sats': 'origination', 
    'num_tl_120dpd_2m': 'origination', 
    'num_tl_30dpd': 'origination', 
    'num_tl_90g_dpd_24m': 'origination', 
    'num_tl_op_past_12m': 'origination', 
    'pct_tl_nvr_dlq': 'origination', 
    'percent_bc_gt_75': 'origination', 
    'pub_rec_bankruptcies': 'origination', 
    'tax_liens': 'origination', 
    'tot_hi_cred_lim': 'origination', 
    'total_bal_ex_mort': 'origination', 
    'total_bc_limit': 'origination', 
    'total_il_high_credit_limit': 'origination', 
    'revol_bal_joint': 'origination', 
    'sec_app_fico_range_low': 'origination', 
    'sec_app_fico_range_high': 'origination', 
    'sec_app_earliest_cr_line': 'origination', 
    'sec_app_inq_last_6mths': 'origination', 
    'sec_app_mort_acc': 'origination', 
    'sec_app_open_acc': 'origination', 
    'sec_app_revol_util': 'origination', 
    'sec_app_open_act_il': 'origination', 
    'sec_app_num_rev_accts': 'origination', 
    'sec_app_chargeoff_within_12_mths': 'origination', 
    'sec_app_collections_12_mths_ex_med': 'origination', 
    'hardship_flag': 'post_origination', 
    'hardship_type': 'post_origination', 
    'hardship_reason': 'post_origination', 
    'hardship_status': 'post_origination', 
    'deferral_term': 'post_origination', 
    'hardship_amount': 'post_origination', 
    'hardship_start_date': 'post_origination', 
    'hardship_end_date': 'post_origination', 
    'payment_plan_start_date': 'post_origination', 
    'hardship_length': 'post_origination', 
    'hardship_dpd': 'post_origination', 
    'hardship_loan_status': 'post_origination', 
    'orig_projected_additional_accrued_interest': 'post_origination', 
    'hardship_payoff_balance_amount': 'post_origination', 
    'hardship_last_payment_amount': 'post_origination', 
    'debt_settlement_flag': 'outcome'
}

In [13]:
data_profile = (
    data_profile
    .with_columns(
        pl.col("feature").replace(AVAILABILITY_MAPPING).alias("availability")
    )
    .select([
        "feature",
        "description",
        "category",
        "semantic_type",
        "availability",
        "dtype",
        "count",
        "null_count",
        "null_pct",
        "unique_count",
        "unique_sample",
        "mean",
        "median",
        "mode",
    ])
)

data_profile.head(5)

feature,description,category,semantic_type,availability,dtype,count,null_count,null_pct,unique_count,unique_sample,mean,median,mode
str,str,str,str,str,str,i64,i64,f64,i64,list[str],f64,f64,str
"""""",null,"""metadata""","""count""","""origination""","""Int64""",2925493,0,0.0,421095,"[""0"", ""1"", … ""4""]",86974.19494,68925.0,"""8944"""
"""id""","""A unique LC assigned ID for the loan listing.""","""metadata""","""identifier""","""origination""","""Int64""",2925493,1,0.000034,2925493,"[""54734"", ""55521"", … ""56121""]",9.7830e7,1.0761e8,"""141921161"""
"""loan_amnt""","""The listed amount of the loan applied for by the borrower. If at some point in time, the credit depa…","""loan_info""","""monetary""","""origination""","""Int64""",2925493,1,0.000034,1573,"[""500"", ""550"", … ""725""]",15358.775105,13000.0,"""10000"""
"""funded_amnt""","""The total amount committed to that loan at that point in time.""","""loan_info""","""monetary""","""origination""","""Int64""",2925493,1,0.000034,1573,"[""500"", ""550"", … ""725""]",15354.704909,13000.0,"""10000"""
"""funded_amnt_inv""","""The total amount committed by investors for that loan at that point in time.""","""loan_info""","""monetary""","""origination""","""Float64""",2925493,1,0.000034,10064,"[""0.0"", ""0.000121098108"", … ""0.000531133069""]",15340.046287,13000.0,"""10000"""


### 2.4.4. PD Eligibility

`pd_eligible` indicates whether a feature is appropriate for use as an input to the **origination-time Probability of Default (PD) model**.

| Value   | Definition                                                                           |
| ------- | ------------------------------------------------------------------------------------ |
| `True`  | Available at origination and suitable as a predictor of future default.              |
| `False` | Unavailable at origination, represents an outcome, or introduces future information. |

**Assignment principle:** A feature is PD eligible only if its information would have been available when the lending decision was made and it does not depend on future loan performance.


In [14]:
# mapping pd eligible

PD_ELIGIBLE_MAPPING = {
    '': False, 
    'id': False, 
    'loan_amnt': True, 
    'funded_amnt': True, 
    'funded_amnt_inv': True, 
    'term': True, 
    'int_rate': True, 
    'installment': True, 
    'grade': True, 
    'sub_grade': True, 
    'emp_title': False, 
    'emp_length': True, 
    'home_ownership': True, 
    'annual_inc': True, 
    'verification_status': True, 
    'issue_d': False, 
    'loan_status': False, 
    'pymnt_plan': False, 
    'url': False, 
    'purpose': True, 
    'title': False, 
    'zip_code': True, 
    'addr_state': True, 
    'dti': True, 
    'delinq_2yrs': True, 
    'earliest_cr_line': True, 
    'fico_range_low': True, 
    'fico_range_high': True, 
    'inq_last_6mths': True, 
    'mths_since_last_delinq': True, 
    'mths_since_last_record': True, 
    'open_acc': True, 
    'pub_rec': True, 
    'revol_bal': True, 
    'revol_util': True, 
    'total_acc': True, 
    'initial_list_status': True, 
    'out_prncp': False, 
    'out_prncp_inv': False, 
    'total_pymnt': False, 
    'total_pymnt_inv': False, 
    'total_rec_prncp': False, 
    'total_rec_int': False, 
    'total_rec_late_fee': False, 
    'recoveries': False, 
    'collection_recovery_fee': False, 
    'last_pymnt_d': False, 
    'last_pymnt_amnt': False, 
    'next_pymnt_d': False, 
    'last_credit_pull_d': False, 
    'last_fico_range_high': False, 
    'last_fico_range_low': False, 
    'collections_12_mths_ex_med': True, 
    'mths_since_last_major_derog': True, 
    'policy_code': False, 
    'application_type': True, 
    'annual_inc_joint': True, 
    'dti_joint': True, 
    'verification_status_joint': True, 
    'acc_now_delinq': True, 
    'tot_coll_amt': True, 
    'tot_cur_bal': True, 
    'open_acc_6m': True, 
    'open_act_il': True, 
    'open_il_12m': True, 
    'open_il_24m': True, 
    'mths_since_rcnt_il': True, 
    'total_bal_il': True, 
    'il_util': True, 
    'open_rv_12m': True, 
    'open_rv_24m': True, 
    'max_bal_bc': True, 
    'all_util': True, 
    'total_rev_hi_lim': True, 
    'inq_fi': True, 
    'total_cu_tl': True, 
    'inq_last_12m': True, 
    'acc_open_past_24mths': True, 
    'avg_cur_bal': True, 
    'bc_open_to_buy': True, 
    'bc_util': True, 
    'chargeoff_within_12_mths': True, 
    'delinq_amnt': True, 
    'mo_sin_old_il_acct': True, 
    'mo_sin_old_rev_tl_op': True, 
    'mo_sin_rcnt_rev_tl_op': True, 
    'mo_sin_rcnt_tl': True, 
    'mort_acc': True, 
    'mths_since_recent_bc': True, 
    'mths_since_recent_bc_dlq': True, 
    'mths_since_recent_inq': True, 
    'mths_since_recent_revol_delinq': True, 
    'num_accts_ever_120_pd': True, 
    'num_actv_bc_tl': True, 
    'num_actv_rev_tl': True, 
    'num_bc_sats': True, 
    'num_bc_tl': True, 
    'num_il_tl': True, 
    'num_op_rev_tl': True, 
    'num_rev_accts': True, 
    'num_rev_tl_bal_gt_0': True, 
    'num_sats': True, 
    'num_tl_120dpd_2m': True, 
    'num_tl_30dpd': True, 
    'num_tl_90g_dpd_24m': True, 
    'num_tl_op_past_12m': True, 
    'pct_tl_nvr_dlq': True, 
    'percent_bc_gt_75': True, 
    'pub_rec_bankruptcies': True, 
    'tax_liens': True, 
    'tot_hi_cred_lim': True, 
    'total_bal_ex_mort': True, 
    'total_bc_limit': True, 
    'total_il_high_credit_limit': True, 
    'revol_bal_joint': True, 
    'sec_app_fico_range_low': True, 
    'sec_app_fico_range_high': True, 
    'sec_app_earliest_cr_line': True, 
    'sec_app_inq_last_6mths': True, 
    'sec_app_mort_acc': True, 
    'sec_app_open_acc': True, 
    'sec_app_revol_util': True, 
    'sec_app_open_act_il': True, 
    'sec_app_num_rev_accts': True, 
    'sec_app_chargeoff_within_12_mths': True, 
    'sec_app_collections_12_mths_ex_med': True, 
    'hardship_flag': False, 
    'hardship_type': False, 
    'hardship_reason': False, 
    'hardship_status': False, 
    'deferral_term': False, 
    'hardship_amount': False, 
    'hardship_start_date': False, 
    'hardship_end_date': False, 
    'payment_plan_start_date': False, 
    'hardship_length': False, 
    'hardship_dpd': False, 
    'hardship_loan_status': False, 
    'orig_projected_additional_accrued_interest': False, 
    'hardship_payoff_balance_amount': False, 
    'hardship_last_payment_amount': False, 
    'debt_settlement_flag': False
}

In [15]:
data_profile = (
    data_profile
    .with_columns(
        pl.col("feature").replace(PD_ELIGIBLE_MAPPING).alias("pd_eligible")
    )
    .select([
        "feature",
        "description",
        "category",
        "semantic_type",
        "availability",
        "pd_eligible",
        "dtype",
        "count",
        "null_count",
        "null_pct",
        "unique_count",
        "unique_sample",
        "mean",
        "median",
        "mode",
    ])
)

data_profile.head(5)

feature,description,category,semantic_type,availability,pd_eligible,dtype,count,null_count,null_pct,unique_count,unique_sample,mean,median,mode
str,str,str,str,str,str,str,i64,i64,f64,i64,list[str],f64,f64,str
"""""",null,"""metadata""","""count""","""origination""","""false""","""Int64""",2925493,0,0.0,421095,"[""0"", ""1"", … ""4""]",86974.19494,68925.0,"""8944"""
"""id""","""A unique LC assigned ID for the loan listing.""","""metadata""","""identifier""","""origination""","""false""","""Int64""",2925493,1,0.000034,2925493,"[""54734"", ""55521"", … ""56121""]",9.7830e7,1.0761e8,"""141921161"""
"""loan_amnt""","""The listed amount of the loan applied for by the borrower. If at some point in time, the credit depa…","""loan_info""","""monetary""","""origination""","""true""","""Int64""",2925493,1,0.000034,1573,"[""500"", ""550"", … ""725""]",15358.775105,13000.0,"""10000"""
"""funded_amnt""","""The total amount committed to that loan at that point in time.""","""loan_info""","""monetary""","""origination""","""true""","""Int64""",2925493,1,0.000034,1573,"[""500"", ""550"", … ""725""]",15354.704909,13000.0,"""10000"""
"""funded_amnt_inv""","""The total amount committed by investors for that loan at that point in time.""","""loan_info""","""monetary""","""origination""","""true""","""Float64""",2925493,1,0.000034,10064,"[""0.0"", ""0.000121098108"", … ""0.000531133069""]",15340.046287,13000.0,"""10000"""


### 2.4.5. Leakage Risk

`leakage_risk` assesses the likelihood that a feature contains information that would not have been available at the PD model's prediction point.

| Level    | Definition                                                                                                     |
| -------- | -------------------------------------------------------------------------------------------------------------- |
| `Low`    | Available at origination with no apparent future information.                                                  |
| `Medium` | Timing or interpretation is potentially ambiguous and requires validation.                                     |
| `High`   | Generated after origination, reflects subsequent performance, or directly contains future outcome information. |

**Assignment principle:** Leakage risk is assessed based on the feature's information timing and relationship to the target, not its predictive strength.


In [16]:
# mapping leakage risk

LEAKAGE_RISK_MAPPING ={
    '': 'low', 
    'id': 'low', 
    'loan_amnt': 'low', 
    'funded_amnt': 'medium', 
    'funded_amnt_inv': 'medium', 
    'term': 'low', 
    'int_rate': 'medium', 
    'installment': 'medium', 
    'grade': 'medium', 
    'sub_grade': 'medium', 
    'emp_title': 'low', 
    'emp_length': 'low', 
    'home_ownership': 'low', 
    'annual_inc': 'low', 
    'verification_status': 'medium', 
    'issue_d': 'low', 
    'loan_status': 'high', 
    'pymnt_plan': 'high', 
    'url': 'low', 
    'purpose': 'low', 
    'title': 'low', 
    'zip_code': 'low', 
    'addr_state': 'low', 
    'dti': 'low', 
    'delinq_2yrs': 'low', 
    'earliest_cr_line': 'low', 
    'fico_range_low': 'low', 
    'fico_range_high': 'low', 
    'inq_last_6mths': 'low', 
    'mths_since_last_delinq': 'high', 
    'mths_since_last_record': 'high', 
    'open_acc': 'low', 
    'pub_rec': 'low', 
    'revol_bal': 'low', 
    'revol_util': 'low', 
    'total_acc': 'low', 
    'initial_list_status': 'medium', 
    'out_prncp': 'high', 
    'out_prncp_inv': 'high', 
    'total_pymnt': 'high', 
    'total_pymnt_inv': 'high', 
    'total_rec_prncp': 'high', 
    'total_rec_int': 'high', 
    'total_rec_late_fee': 'high', 
    'recoveries': 'high', 
    'collection_recovery_fee': 'high', 
    'last_pymnt_d': 'high', 
    'last_pymnt_amnt': 'high', 
    'next_pymnt_d': 'high', 
    'last_credit_pull_d': 'high', 
    'last_fico_range_high': 'high', 
    'last_fico_range_low': 'high', 
    'collections_12_mths_ex_med': 'high', 
    'mths_since_last_major_derog': 'high', 
    'policy_code': 'low', 
    'application_type': 'low', 
    'annual_inc_joint': 'medium', 
    'dti_joint': 'medium', 
    'verification_status_joint': 'medium', 
    'acc_now_delinq': 'high', 
    'tot_coll_amt': 'low', 
    'tot_cur_bal': 'low', 
    'open_acc_6m': 'low', 
    'open_act_il': 'low', 
    'open_il_12m': 'low', 
    'open_il_24m': 'low', 
    'mths_since_rcnt_il': 'low', 
    'total_bal_il': 'low', 
    'il_util': 'low', 
    'open_rv_12m': 'low', 
    'open_rv_24m': 'low', 
    'max_bal_bc': 'low', 
    'all_util': 'low', 
    'total_rev_hi_lim': 'low', 
    'inq_fi': 'low', 
    'total_cu_tl': 'low', 
    'inq_last_12m': 'low', 
    'acc_open_past_24mths': 'low', 
    'avg_cur_bal': 'low', 
    'bc_open_to_buy': 'low', 
    'bc_util': 'low', 
    'chargeoff_within_12_mths': 'high', 
    'delinq_amnt': 'high', 
    'mo_sin_old_il_acct': 'low', 
    'mo_sin_old_rev_tl_op': 'low', 
    'mo_sin_rcnt_rev_tl_op': 'low', 
    'mo_sin_rcnt_tl': 'low', 
    'mort_acc': 'low', 
    'mths_since_recent_bc': 'low', 
    'mths_since_recent_bc_dlq': 'high', 
    'mths_since_recent_inq': 'low', 
    'mths_since_recent_revol_delinq': 'high', 
    'num_accts_ever_120_pd': 'low', 
    'num_actv_bc_tl': 'low', 
    'num_actv_rev_tl': 'low', 
    'num_bc_sats': 'low', 
    'num_bc_tl': 'low', 
    'num_il_tl': 'low', 
    'num_op_rev_tl': 'low', 
    'num_rev_accts': 'low', 
    'num_rev_tl_bal_gt_0': 'low', 
    'num_sats': 'low', 
    'num_tl_120dpd_2m': 'high', 
    'num_tl_30dpd': 'high', 
    'num_tl_90g_dpd_24m': 'high', 
    'num_tl_op_past_12m': 'low', 
    'pct_tl_nvr_dlq': 'low', 
    'percent_bc_gt_75': 'low', 
    'pub_rec_bankruptcies': 'low', 
    'tax_liens': 'low', 
    'tot_hi_cred_lim': 'low', 
    'total_bal_ex_mort': 'low', 
    'total_bc_limit': 'low', 
    'total_il_high_credit_limit': 'low', 
    'revol_bal_joint': 'medium', 
    'sec_app_fico_range_low': 'low', 
    'sec_app_fico_range_high': 'low', 
    'sec_app_earliest_cr_line': 'low', 
    'sec_app_inq_last_6mths': 'low', 
    'sec_app_mort_acc': 'low', 
    'sec_app_open_acc': 'low', 
    'sec_app_revol_util': 'low', 
    'sec_app_open_act_il': 'low', 
    'sec_app_num_rev_accts': 'low', 
    'sec_app_chargeoff_within_12_mths': 'high', 
    'sec_app_collections_12_mths_ex_med': 'high', 
    'hardship_flag': 'high', 
    'hardship_type': 'high', 
    'hardship_reason': 'high', 
    'hardship_status': 'high', 
    'deferral_term': 'high', 
    'hardship_amount': 'high', 
    'hardship_start_date': 'high', 
    'hardship_end_date': 'high', 
    'payment_plan_start_date': 'high', 
    'hardship_length': 'high', 
    'hardship_dpd': 'high', 
    'hardship_loan_status': 'high', 
    'orig_projected_additional_accrued_interest': 'high', 
    'hardship_payoff_balance_amount': 'high', 
    'hardship_last_payment_amount': 'high', 
    'debt_settlement_flag': 'high'
}

In [17]:
data_profile = (
    data_profile
    .with_columns(
        pl.col("feature").replace(LEAKAGE_RISK_MAPPING).alias("leakage_risk")
    )
    .select([
        "feature",
        "description",
        "category",
        "semantic_type",
        "availability",
        "pd_eligible",
        "leakage_risk",
        "dtype",
        "count",
        "null_count",
        "null_pct",
        "unique_count",
        "unique_sample",
        "mean",
        "median",
        "mode",
    ])
)

data_profile.head(5)

feature,description,category,semantic_type,availability,pd_eligible,leakage_risk,dtype,count,null_count,null_pct,unique_count,unique_sample,mean,median,mode
str,str,str,str,str,str,str,str,i64,i64,f64,i64,list[str],f64,f64,str
"""""",null,"""metadata""","""count""","""origination""","""false""","""low""","""Int64""",2925493,0,0.0,421095,"[""0"", ""1"", … ""4""]",86974.19494,68925.0,"""8944"""
"""id""","""A unique LC assigned ID for the loan listing.""","""metadata""","""identifier""","""origination""","""false""","""low""","""Int64""",2925493,1,0.000034,2925493,"[""54734"", ""55521"", … ""56121""]",9.7830e7,1.0761e8,"""141921161"""
"""loan_amnt""","""The listed amount of the loan applied for by the borrower. If at some point in time, the credit depa…","""loan_info""","""monetary""","""origination""","""true""","""low""","""Int64""",2925493,1,0.000034,1573,"[""500"", ""550"", … ""725""]",15358.775105,13000.0,"""10000"""
"""funded_amnt""","""The total amount committed to that loan at that point in time.""","""loan_info""","""monetary""","""origination""","""true""","""medium""","""Int64""",2925493,1,0.000034,1573,"[""500"", ""550"", … ""725""]",15354.704909,13000.0,"""10000"""
"""funded_amnt_inv""","""The total amount committed by investors for that loan at that point in time.""","""loan_info""","""monetary""","""origination""","""true""","""medium""","""Float64""",2925493,1,0.000034,10064,"[""0.0"", ""0.000121098108"", … ""0.000531133069""]",15340.046287,13000.0,"""10000"""


## 2.5. Data Availability

In [18]:
# get all columns except the time key
all_cols = [c for c in lf.collect_schema().names() if c not in ("issue_d", "issue_month_year")]

# build null aggs dynamically
null_aggs = [
    pl.col(c).null_count().alias(f"{c}_nulls")
    for c in all_cols
]

# build vintage profile
vintage_profile = (
    lf
    .with_columns(
        pl.col("issue_d")
        .str.strptime(pl.Date, "%b-%Y", strict=False)
        .dt.strftime("%Y-%m")
        .alias("issue_month_year")
    )
    .group_by("issue_month_year")
    .agg([
        pl.len().alias("n_loans"),
        *null_aggs,
    ])
    .sort("issue_month_year")
    .collect()
)

# get null columns, drop issue_month_year_nulls
null_cols = [
    c for c in vintage_profile.columns
    if c.endswith("_nulls") and c != "issue_month_year_nulls"
]

In [19]:
# total column count for null rate denominator
total_cols = len(all_cols)

# compute per month, % of columns with nulls + avg null rate
completeness_profile = (
    vintage_profile
    .drop_nulls()
    .with_columns([
        # how many columns have ANY null this month
        pl.sum_horizontal([
            (pl.col(c) > 0).cast(pl.Int8) for c in null_cols
        ]).alias("n_cols_with_nulls"),

        # average null rate across all columns this month
        pl.mean_horizontal([
            (pl.col(c) / pl.col("n_loans")) for c in null_cols
        ]).alias("avg_null_rate"),
    ])
    .with_columns([
        # % of columns that are dirty this month
        (pl.col("n_cols_with_nulls") / total_cols * 100).alias("pct_cols_with_nulls"),
    ])
    .select([
        "issue_month_year",
        "n_loans",
        "n_cols_with_nulls",
        "pct_cols_with_nulls",
        "avg_null_rate",
    ])
)

completeness_profile

issue_month_year,n_loans,n_cols_with_nulls,pct_cols_with_nulls,avg_null_rate
str,u32,i8,f64,f64
"""2007-06""",24,98,69.503546,0.666076
"""2007-07""",63,99,70.212766,0.605764
"""2007-08""",74,99,70.212766,0.595553
"""2007-09""",53,88,62.411348,0.579419
"""2007-10""",105,87,61.702128,0.578116
"""2007-11""",112,83,58.865248,0.577191
"""2007-12""",172,84,59.574468,0.578179
"""2008-01""",305,83,58.865248,0.578398
"""2008-02""",306,84,59.574468,0.578431


In [20]:
import pandas as pd

df_plot = completeness_profile.to_pandas()

# convert to datetime so add_vline works correctly
df_plot["issue_month_year"] = pd.to_datetime(df_plot["issue_month_year"])
cutoff = pd.to_datetime("2016-01")

# --- plot 1: % of columns with nulls ---
fig1 = px.line(
    df_plot,
    x="issue_month_year",
    y="pct_cols_with_nulls",
    markers=True,
    title="% of columns with at least one null",
    labels={"issue_month_year": "Month-Year", "pct_cols_with_nulls": "% cols with nulls"},
)
fig1.add_vline(x=cutoff, line_dash="dash", line_color="red", line_width=1.5,
               annotation_text="Jan 2016 cutoff", annotation_position="top right")
fig1.update_layout(hovermode="x unified")
fig1.show()

# --- plot 2: average null rate ---
df_plot["avg_null_pct"] = df_plot["avg_null_rate"] * 100

fig2 = px.line(
    df_plot,
    x="issue_month_year",
    y="avg_null_pct",
    markers=True,
    title="Average null rate across all columns (%)",
    labels={"issue_month_year": "Month-Year", "avg_null_pct": "avg null rate (%)"},
)
fig2.add_vline(x=cutoff, line_dash="dash", line_color="red", line_width=1.5,
               annotation_text="Jan 2016 cutoff", annotation_position="top right")
fig2.update_layout(hovermode="x unified")
fig2.show()

# --- plot 3: loan volume ---
fig3 = px.bar(
    df_plot,
    x="issue_month_year",
    y="n_loans",
    title="Number of loans issued per month",
    labels={"issue_month_year": "Month-Year", "n_loans": "n loans"},
    color_discrete_sequence=["#1baf7a"],
)
fig3.add_vline(x=cutoff, line_dash="dash", line_color="red", line_width=1.5,
               annotation_text="Jan 2016 cutoff", annotation_position="top right")
fig3.update_layout(hovermode="x unified")
fig3.show()

We selected January 2016 as the analysis start date based on three quantitative criteria observed in the LendingClub 2007–2020 dataset:

1. Column completeness, the share of columns containing at least one null dropped from 41%–70% (pre-2016) to a stable ~37% from January 2016 onwards, reflecting more consistent data recording practices.

2. Average null rate, monthly null rates compressed from a volatile 28%–66% range to a stable ~23%, suggesting the earlier period reflects platform data immaturity rather than genuine borrower risk signals.

3. Loan volume, sufficient monthly volume (>2.755 loans) is consistently present from January 2016, reducing sampling noise in month-level aggregations.

Note: the residual ~23% null rate post-2016 is driven primarily by post-origination and target-related fields that will be excluded from the feature set on data leakage grounds. Features are restricted to information available at loan application time, after which effective missingness is substantially lower.

## 2.6. Missingness

In [21]:
data_profile.columns

['feature',
 'description',
 'category',
 'semantic_type',
 'availability',
 'pd_eligible',
 'leakage_risk',
 'dtype',
 'count',
 'null_count',
 'null_pct',
 'unique_count',
 'unique_sample',
 'mean',
 'median',
 'mode']

In [22]:
# show all rows in notebook
pl.Config.set_tbl_rows(-1)

(data_profile
    .filter(pl.col("null_pct") > 90)
    .select(["feature","category", "null_count","null_pct"])
    .sort("null_pct", descending=True))

feature,category,null_count,null_pct
str,str,i64,f64
"""hardship_loan_status""","""hardship""",2782082,95.097886
"""hardship_reason""","""hardship""",2781861,95.090332
"""hardship_status""","""hardship""",2781858,95.090229
"""hardship_dpd""","""hardship""",2781856,95.090161
"""hardship_type""","""hardship""",2781855,95.090127
"""deferral_term""","""hardship""",2781855,95.090127
"""hardship_start_date""","""hardship""",2781855,95.090127
"""hardship_end_date""","""hardship""",2781855,95.090127
"""payment_plan_start_date""","""hardship""",2781855,95.090127


In [23]:
(data_profile
    .filter([pl.col("null_pct") > 10, pl.col("null_pct") <= 90])
    .select(["feature","category", "null_count","null_pct"])
    .sort("null_pct", descending=True))

feature,category,null_count,null_pct
str,str,i64,f64
"""mths_since_last_record""","""public_record""",2498045,85.388856
"""mths_since_recent_bc_dlq""","""delinquency""",2274851,77.759578
"""mths_since_last_major_derog""","""delinquency""",2202576,75.289054
"""mths_since_recent_revol_delinq""","""delinquency""",1994887,68.189772
"""next_pymnt_d""","""payment_history""",1860332,63.590376
"""mths_since_last_delinq""","""delinquency""",1536503,52.521165
"""il_util""","""installment_credit""",1157149,39.553983
"""mths_since_rcnt_il""","""installment_credit""",926530,31.670901
"""all_util""","""credit_profile""",866486,29.618461


In [24]:
(data_profile
    .filter([pl.col("null_pct") > 1, pl.col("null_pct") <= 10])
    .select(["feature","category", "null_count","null_pct"])
    .sort("null_pct", descending=True))

feature,category,null_count,null_pct
str,str,i64,f64
"""emp_title""","""borrower_profile""",264087,9.027094
"""emp_length""","""borrower_profile""",205221,7.01492
"""num_tl_120dpd_2m""","""delinquency""",161687,5.526829
"""mo_sin_old_il_acct""","""installment_credit""",155677,5.321394
"""bc_util""","""revolving_credit""",84142,2.876165
"""percent_bc_gt_75""","""revolving_credit""",83150,2.842256
"""bc_open_to_buy""","""revolving_credit""",82669,2.825814
"""mths_since_recent_bc""","""revolving_credit""",80702,2.758578
"""pct_tl_nvr_dlq""","""credit_profile""",70432,2.407526


In [25]:
(data_profile
    .filter([pl.col("null_pct") <= 1])
    .select(["feature", "category", "null_count","null_pct"])
    .sort("null_pct", descending=True))

feature,category,null_count,null_pct
str,str,i64,f64
"""title""","""loan_info""",23326,0.797336
"""last_pymnt_d""","""payment_history""",4922,0.168245
"""dti""","""credit_profile""",3109,0.106273
"""revol_util""","""revolving_credit""",2661,0.090959
"""pub_rec_bankruptcies""","""public_record""",1366,0.046693
"""collections_12_mths_ex_med""","""delinquency""",146,0.004991
"""chargeoff_within_12_mths""","""delinquency""",146,0.004991
"""tax_liens""","""public_record""",106,0.003623
"""last_credit_pull_d""","""credit_history""",76,0.002598


In [26]:
# show 20 rows in notebook
pl.Config.set_tbl_rows(20)

polars.config.Config

An initial missingness audit across all 142 columns revealed that the highest null rates are concentrated in two structural categories:

1. Hardship fields (e.g. hardship_status, hardship_reason) with 93%–95% missing, as these are only populated for the small subset of borrowers who entered a hardship programme.

2. Joint application fields (e.g. annual_inc_joint, dti_joint, sec_app_fico_range) with 92%–93% missing, reflecting that the majority of LendingClub loans are individual rather than joint applications.

Both categories will be excluded from the feature set, as hardship fields are post-origination by nature and joint application fields are irrelevant for individual loan applicants. For remaining features, missingness will be assessed within the January 2016 onwards window, where columns with substantial null counts will be dropped. Given that data recording had stabilised by this point, residual high missingness is more likely to reflect a structurally sparse field than a historical data maturity issue, making exclusion more appropriate than imputation.

## 2.7. Data Quality and Type Assessment

In [27]:
# mapping cast

CAST_MAPPING = {
    '': 'Int64', 
    'id': 'Int64', 
    'loan_amnt': 'Int64', 
    'funded_amnt': 'Int64', 
    'funded_amnt_inv': 'Float64', 
    'term': 'Int64', 
    'int_rate': 'Float64', 
    'installment': 'Float64', 
    'grade': 'category', 
    'sub_grade': 'category', 
    'emp_title': 'string', 
    'emp_length': 'Int64', 
    'home_ownership': 'category', 
    'annual_inc': 'Float64', 
    'verification_status': 'category', 
    'issue_d': 'datetime64[ns]', 
    'loan_status': 'category', 
    'pymnt_plan': 'category', 
    'url': 'string', 
    'purpose': 'category', 
    'title': 'string', 
    'zip_code': 'string', 
    'addr_state': 'category', 
    'dti': 'Float64', 
    'delinq_2yrs': 'Int64', 
    'earliest_cr_line': 'datetime64[ns]', 
    'fico_range_low': 'Int64', 
    'fico_range_high': 'Int64', 
    'inq_last_6mths': 'Int64', 
    'mths_since_last_delinq': 'Int64', 
    'mths_since_last_record': 'Int64', 
    'open_acc': 'Int64', 
    'pub_rec': 'Int64', 
    'revol_bal': 'Float64', 
    'revol_util': 'Float64', 
    'total_acc': 'Int64', 
    'initial_list_status': 'category', 
    'out_prncp': 'Float64', 
    'out_prncp_inv': 'Float64', 
    'total_pymnt': 'Float64', 
    'total_pymnt_inv': 'Float64', 
    'total_rec_prncp': 'Float64', 
    'total_rec_int': 'Float64', 
    'total_rec_late_fee': 'Float64', 
    'recoveries': 'Float64', 
    'collection_recovery_fee': 'Float64', 
    'last_pymnt_d': 'datetime64[ns]', 
    'last_pymnt_amnt': 'Float64', 
    'next_pymnt_d': 'datetime64[ns]', 
    'last_credit_pull_d': 'datetime64[ns]', 
    'last_fico_range_high': 'Int64', 
    'last_fico_range_low': 'Int64', 
    'collections_12_mths_ex_med': 'Int64', 
    'mths_since_last_major_derog': 'Int64', 
    'policy_code': 'category', 
    'application_type': 'category', 
    'annual_inc_joint': 'Float64', 
    'dti_joint': 'Float64', 
    'verification_status_joint': 'category', 
    'acc_now_delinq': 'Int64', 
    'tot_coll_amt': 'Float64', 
    'tot_cur_bal': 'Float64', 
    'open_acc_6m': 'Int64', 
    'open_act_il': 'Int64', 
    'open_il_12m': 'Int64', 
    'open_il_24m': 'Int64', 
    'mths_since_rcnt_il': 'Int64', 
    'total_bal_il': 'Float64', 
    'il_util': 'Float64', 
    'open_rv_12m': 'Int64', 
    'open_rv_24m': 'Int64', 
    'max_bal_bc': 'Float64', 
    'all_util': 'Float64', 
    'total_rev_hi_lim': 'Float64', 
    'inq_fi': 'Int64', 
    'total_cu_tl': 'Int64', 
    'inq_last_12m': 'Int64', 
    'acc_open_past_24mths': 'Int64', 
    'avg_cur_bal': 'Float64', 
    'bc_open_to_buy': 'Float64', 
    'bc_util': 'Float64', 
    'chargeoff_within_12_mths': 'Int64', 
    'delinq_amnt': 'Float64', 
    'mo_sin_old_il_acct': 'Int64', 
    'mo_sin_old_rev_tl_op': 'Int64', 
    'mo_sin_rcnt_rev_tl_op': 'Int64', 
    'mo_sin_rcnt_tl': 'Int64', 
    'mort_acc': 'Int64', 
    'mths_since_recent_bc': 'Int64', 
    'mths_since_recent_bc_dlq': 'Int64', 
    'mths_since_recent_inq': 'Int64', 
    'mths_since_recent_revol_delinq': 'Int64', 
    'num_accts_ever_120_pd': 'Int64', 
    'num_actv_bc_tl': 'Int64', 
    'num_actv_rev_tl': 'Int64', 
    'num_bc_sats': 'Int64', 
    'num_bc_tl': 'Int64', 
    'num_il_tl': 'Int64', 
    'num_op_rev_tl': 'Int64', 
    'num_rev_accts': 'Int64', 
    'num_rev_tl_bal_gt_0': 'Int64', 
    'num_sats': 'Int64', 
    'num_tl_120dpd_2m': 'Int64', 
    'num_tl_30dpd': 'Int64', 
    'num_tl_90g_dpd_24m': 'Int64', 
    'num_tl_op_past_12m': 'Int64', 
    'pct_tl_nvr_dlq': 'Float64', 
    'percent_bc_gt_75': 'Float64', 
    'pub_rec_bankruptcies': 'Int64', 
    'tax_liens': 'Int64', 
    'tot_hi_cred_lim': 'Float64', 
    'total_bal_ex_mort': 'Float64', 
    'total_bc_limit': 'Float64', 
    'total_il_high_credit_limit': 'Float64', 
    'revol_bal_joint': 'Float64', 
    'sec_app_fico_range_low': 'Int64', 
    'sec_app_fico_range_high': 'Int64', 
    'sec_app_earliest_cr_line': 'datetime64[ns]', 
    'sec_app_inq_last_6mths': 'Int64', 
    'sec_app_mort_acc': 'Int64', 
    'sec_app_open_acc': 'Int64', 
    'sec_app_revol_util': 'Float64', 
    'sec_app_open_act_il': 'Int64', 
    'sec_app_num_rev_accts': 'Int64', 
    'sec_app_chargeoff_within_12_mths': 'Int64', 
    'sec_app_collections_12_mths_ex_med': 'Int64', 
    'hardship_flag': 'category', 
    'hardship_type': 'category', 
    'hardship_reason': 'category', 
    'hardship_status': 'category', 
    'deferral_term': 'Int64', 
    'hardship_amount': 'Float64', 
    'hardship_start_date': 'datetime64[ns]', 
    'hardship_end_date': 'datetime64[ns]', 
    'payment_plan_start_date': 'datetime64[ns]', 
    'hardship_length': 'Int64', 
    'hardship_dpd': 'Int64', 
    'hardship_loan_status': 'category', 
    'orig_projected_additional_accrued_interest': 'Float64', 
    'hardship_payoff_balance_amount': 'Float64', 
    'hardship_last_payment_amount': 'Float64', 
    'debt_settlement_flag': 'category'
}

In [28]:
data_profile = (
    data_profile
    .with_columns(
        pl.col("feature").replace(CAST_MAPPING).alias("cast")
    )
    .select([
        "feature",
        "description",
        "category",
        "semantic_type",
        "availability",
        "pd_eligible",
        "leakage_risk",
        "dtype",
        "cast",
        "count",
        "null_count",
        "null_pct",
        "unique_count",
        "unique_sample",
        "mean",
        "median",
        "mode",
    ])
)

data_profile.head(5)

feature,description,category,semantic_type,availability,pd_eligible,leakage_risk,dtype,cast,count,null_count,null_pct,unique_count,unique_sample,mean,median,mode
str,str,str,str,str,str,str,str,str,i64,i64,f64,i64,list[str],f64,f64,str
"""""",null,"""metadata""","""count""","""origination""","""false""","""low""","""Int64""","""Int64""",2925493,0,0.0,421095,"[""0"", ""1"", … ""4""]",86974.19494,68925.0,"""8944"""
"""id""","""A unique LC assigned ID for the loan listing.""","""metadata""","""identifier""","""origination""","""false""","""low""","""Int64""","""Int64""",2925493,1,0.000034,2925493,"[""54734"", ""55521"", … ""56121""]",9.7830e7,1.0761e8,"""141921161"""
"""loan_amnt""","""The listed amount of the loan applied for by the borrower. If at some point in time, the credit depa…","""loan_info""","""monetary""","""origination""","""true""","""low""","""Int64""","""Int64""",2925493,1,0.000034,1573,"[""500"", ""550"", … ""725""]",15358.775105,13000.0,"""10000"""
"""funded_amnt""","""The total amount committed to that loan at that point in time.""","""loan_info""","""monetary""","""origination""","""true""","""medium""","""Int64""","""Int64""",2925493,1,0.000034,1573,"[""500"", ""550"", … ""725""]",15354.704909,13000.0,"""10000"""
"""funded_amnt_inv""","""The total amount committed by investors for that loan at that point in time.""","""loan_info""","""monetary""","""origination""","""true""","""medium""","""Float64""","""Float64""",2925493,1,0.000034,10064,"[""0.0"", ""0.000121098108"", … ""0.000531133069""]",15340.046287,13000.0,"""10000"""


## 2.8. Target Definitions

In [29]:
# count loan status
loan_status = (lf
                .group_by("loan_status")
                .len()
                .sort("len", descending=True)
                ).collect()

# visualize
fig = px.bar(
    loan_status,
    x="loan_status",
    y="len",
    title="Loan Status Distribution",
    labels={
        "loan_status": "Loan Status",
        "len": "Number of Loans"
    },
    text_auto=True
)

fig.update_layout(
    template="plotly_white",
    xaxis_tickangle=-45
)

fig.show()

In [30]:
temp = (lf
        .with_columns(
            pl.col("loan_status").replace(
                {"Charged Off" : "default",
                "Default" : "default",
                "Does not meet the credit policy. Status:Charged Off" : "default",
                "Fully Paid" : "paid",
                "Does not meet the credit policy. Status:Fully Paid" : "paid",
                "Current": "not resolve",
                "In Grace Period": "not resolve",
                "Late (16-30 days)": "not resolve",
                "Late (31-120 days)": "not resolve",
                "Issued" : "not resolve"
                }).alias("loan_status_simple").cast(pl.String)))

loan_status = (
    temp
    .group_by("loan_status_simple")
    .len()
    .with_columns(
        (pl.col("len") / pl.col("len").sum() * 100)
        .round(2)
        .alias("percentage")
    )
    .sort("len", descending=True)
    .collect()
)

fig = px.bar(
    loan_status,
    x="loan_status_simple",
    y="len",
    color="loan_status_simple",
    color_discrete_map={
        "default": "#D23131",
        "paid": "#11B81C",
        "not resolve": "#79747D"
    },
    title="Loan Status Distribution",
    labels={
        "loan_status_simple": "Loan Status",
        "len": "Number of Loans"
    },
    text="percentage"
)

fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    xaxis_tickangle=-45
)

fig.show()

In [31]:
monthly = (temp
            .filter(pl.col("loan_status_simple").is_not_null())
            .with_columns((
                pl.lit("01-") + pl.col("issue_d"))
                .str.to_date("%d-%b-%Y")
                .dt.truncate("1mo")
                .alias("issue_month"))
            .group_by("issue_month")
            .agg(pl.len().alias("total"),
                (pl.col("loan_status_simple") == "paid").sum().alias("paid"),
                (pl.col("loan_status_simple") == "default").sum().alias("default"),
                (pl.col("loan_status_simple") == "not resolve").sum().alias("not_resolve"))
            .sort("issue_month")
        ).collect()

fig = px.line(
    monthly,
    x="issue_month",
    y=["total", "paid", "default", "not_resolve"],
    markers=True,
    title="Month - Year Loan Status",
    color_discrete_map={
            "total": "#1f77b4",
            "default": "#D23131",
            "paid": "#11B81C",
            "not resolve": "#79747D"
        },
    labels={
        "issue_month": "Issue Month",
        "value": "Number of Loans",
        "variable": "Loan Outcome"
    }
)

fig.update_layout(
    template="plotly_white",
    hovermode="x unified",
    legend=dict(
        title="Loan Outcome",
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="left",
        x=0
    )
)

fig.update_yaxes(
    type="log",
    title="Number of Loans (log scale)"
)

fig.show()

**Target Variable Analysis and Sample Definition**

- **Target Variable - Full Portfolio Overview**

The raw loan_status field contains 11 distinct values across 2.93M loans, consolidated into three buckets for analysis:

| Bucket       | Raw loan_status values                                                        |
|--------------|-------------------------------------------------------------------------------|
| Paid         | Fully Paid; Does not meet the credit policy. Status: Fully Paid               |
| Default      | Charged Off; Default; Does not meet the credit policy. Status: Charged Off    |
| Not Resolved | Current; In Grace Period; Late (16–30 days); Late (31–120 days); Issued       |

The full portfolio distribution prior to any filtering is as follows:

| Status       | Count     | %      |
|--------------|-----------|--------|
| Paid         | 1,499,771 | 51.27% |
| Default      | 363,742   | 12.43% |
| Not Resolved | 1,061,979 | 36.30% |

The 12.43% default rate is a portfolio-composition figure, not a true default rate. It is diluted by the 36.30% of loans that remain unresolved and have not yet had the opportunity to default. This unresolved share is concentrated in the most recent, highest-volume vintages, with 2019–2020 cohorts exceeding 95% unresolved, driven by the substantial growth in monthly origination volume over the period (approximately 24 loans per month in 2007 rising to 42,000 in 2019).

- **Sample Definition for Modelling**

Two filters are applied to construct the modelling sample from the full portfolio:

| Filter                    | Rationale                                                                                      |
|---------------------------|------------------------------------------------------------------------------------------------|
| loan_status in Paid, Default | Removes censored observations whose final outcome is unknown, eliminating label noise       |
| issue_d from January 2016 | Restricts to the window of stable data recording identified in the missingness analysis above |

Applying the resolved-status filter alone shifts the effective default rate to 363,742 / (363,742 + 1,499,771) = 19.5%, which is the operationally relevant class imbalance figure for modelling rather than the 12.43% observed in the full portfolio.

The January 2016 cutoff serves a dual purpose here: it aligns with the data stability threshold identified earlier, where null rates stabilised and data recording practices matured, and ensures that features selected for modelling benefit from the fuller bureau attribute coverage available from that period onwards. The exact default rate after applying both filters jointly will be computed directly from the dataset in the preprocessing stage.

## 2.9. Data Understanding Summary

**Dataset:** 2.93M loan records, 142 features, LendingClub 2007–2020 Q3.

**Data Quality**
- Pre-2016 null rates were volatile (28%–66%) and inconsistent (41%–70% of columns affected), reflecting platform data immaturity.
- From January 2016 onwards both metrics stabilised, establishing the analysis start date.
- Highest missingness is structural: hardship fields (93%–95%) and joint application fields (92%–93%), both excluded on leakage and relevance grounds.
- Remaining high-null columns within the January 2016 window will be dropped rather than imputed.

**Target Variable**
- loan_status consolidated from 11 raw values into three buckets: Paid (51.27%), Not Resolved (36.30%), Default (12.43%).
- The 12.43% headline default rate is diluted by the 36.30% unresolved share, concentrated in recent high-volume vintages.
- Restricting to resolved loans only shifts the effective default rate to 19.5%, the operationally relevant class imbalance figure.

**Modelling Sample**
- Filter 1: resolved loans only (Paid and Default), removing censored observations with unknown outcomes.
- Filter 2: issue_d from January 2016 onwards, aligning with the stable data recording window and consistent bureau attribute coverage.

**Feature Categorisation**
- Retain: origination features available at loan application time (credit score, income, DTI, grade, bureau attributes).
- Exclude: post-origination features (payment history, balances, recoveries) due to data leakage.
- Exclude: target and target-adjacent fields (loan_status and derived outcome variables).

## 2.10 Other and Save File

In [32]:
data_profile.write_parquet(INTERIM_DIR / "data_profile.parquet")